In [2]:
#import seaborn as sns
import pandas as pd
from pylab import *
%matplotlib inline
#from scipy.stats import gaussian_kde
from matplotlib.colors import ListedColormap
#from scipy.spatial import ConvexHull
#import statistics
#from scipy.spatial.distance import mahalanobis
from preprocessing_aux import *

In [3]:
filename = 'data/RGB_rcgc_chm_dataset.csv'

df = pd.read_csv(filename)
sites = ['CG1-8A', 'CG1-8B', 'CS3A', 'CS3B', 'CS-46A', 'CS-46B', 'CS-59B', 'CS-96B', 'CS-103B', 'CS117B', 
         'F3-8A', 'F3-8B', 'F3-8C', 'F3-20A', 'F3-20B', 'F3-20C', 'F3-19B', 'F3-19C', 
         'ZF20-11A', 'ZF46-15A', 'ZF46-37A']

classnames = ['lichen', 'rock', 'broadleaf', 'needleleaf', 'deadwood', 'graminoids', 
              'moss', 'soil', 'low_green', 'dry_branches']

# --------------------------------------------------------------------------------------
# activate to use a selection of classes instead of all classes:
#options = ['lichen', 'deadwood']
#subdata_df = df[df['veg_class'].isin(options)]

In [3]:
# --------------------------------------------------------------------------------------
# extending pada data frame by rc/gc, rc+gc, luminance Y, perceived lightness L, z_score_Y, z_score_L 
# Excess indecies ExG, ExR, ExB, indices ExGmExR, Ikaw, MGRVI, and GLI 
# --------------------------------------------------------------------------------------
# blue chromaticity
df['bc'] = 1-(df['rc']+df['gc'])

# ration rc/gc
df['rc/gc'] = df['rc']/df['gc']

# sum rc+gc
df['rc+gc'] = df['rc']+df['gc']

# excess indecies
ExG = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False)
       for i, val in enumerate(np.array(df['G']))]
df['ExG'] = ExG

ExR = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='red')
       for i, val in enumerate(np.array(df['G']))]
df['ExR'] = ExR

ExB = [excess_index(G=val, R=np.array(df['R'])[i], B=np.array(df['B'])[i], convert_to_255=False, excessband='blue')
       for i, val in enumerate(np.array(df['G']))]
df['ExB'] = ExB

df.head()

# index ExGmExR
df['ExGmExR'] = df['ExG']-df['ExR']

# index Ikaw - Kawashima Index
df['Ikaw'] = (df['R']-df['B'])/(df['R']+df['B'])

# MGRVI - modified green red vegetation index
df['MGRVI'] = ((df['G'])**2 - (df['R'])**2)/((df['G'])**2 + (df['R'])**2)

# GLI - green leaf index
df['GLI'] = (2*df['G'] - df['R'] - df['B'])/(2*df['G'] + df['R'] + df['B'])

# luminance Y
Y = [0.2126*sRGBtoLin(val) + 
     0.7152*sRGBtoLin(np.array(df['G'])[i]) + 
     0.0722*sRGBtoLin(np.array(df['B'])[i]) for i, val in enumerate(np.array(df['R']))]
df['Y'] = Y

# perceived lightness L
L = [YtoLstar(val) for val in df['Y']]
df['L'] = L

# z_score of the luminance Y
df['z_score_Y'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['Y']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_Y'] = z_site
    
# z_score of the perceived lightness L
df['z_score_L'] = 0
sites = set(df['site'])
for site in sites:
    condition = df['site'] == site
    subdata_site = df[df['site'].isin([site])]['L']
    z_site = z_score(x_all=np.array(subdata_site))
    df.loc[condition, 'z_score_L'] = z_site

print(df.head())
outname = 'data/transformed_chm_data.csv'
#df.to_csv(loadpath+outname, index=False)

     site veg_class         R  ...          L  z_score_Y  z_score_L
0  CG1-8A    lichen  0.862745  ...  86.277954   0.901666   0.875787
1  CG1-8A    lichen  0.909804  ...  91.125221   1.338999   1.162117
2  CG1-8A    lichen  0.894118  ...  89.739293   1.209810   1.080249
3  CG1-8A    lichen  0.909804  ...  91.172679   1.343483   1.164920
4  CG1-8A    lichen  0.937255  ...  93.923383   1.610195   1.327405

[5 rows x 22 columns]
